# Controlled factorial 2×2 — Dev-only Kaggle training

This notebook runs three paired seeds (`2026`, `2126`, `2226`) for the small and expanded weak-label pools. It does not mount, read, or evaluate ViLexNorm Test. Each arm trains to eight epochs without early stopping; Dev-selected horizon-3 and horizon-8 artifacts are exported.

In [ ]:
from pathlib import Path
import shutil, subprocess

REPO = Path('/kaggle/working/VisolexNorm')
SOURCE_REF = 'main'  # replace with the committed controlled-experiment revision
REPOSITORY_URL = 'https://github.com/AIVIETNAM-AIO-DinhBao/VisolexNorm.git'
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', SOURCE_REF, REPOSITORY_URL, str(REPO)], check=True)
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
print('Frozen source revision:', SOURCE_COMMIT)

In [ ]:
%cd {REPO}
!pip install -q -r requirements-kaggle.txt
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
print('GPU:', torch.cuda.get_device_name(0))

## Private input dataset
Attach a private Kaggle Dataset as an unpacked directory containing only `checkpoints/model_a/` and the four processed files required below. Do **not** upload a ZIP: unpacking Model A into `/kaggle/working` duplicates several GB and can exhaust disk. Do **not** upload `vilexnorm_test.jsonl` or `outputs/evaluation/`.

In [ ]:
MOUNT = Path('/kaggle/input/visolexnorm-controlled-input')  # update slug if needed
WORK = Path('/kaggle/working/controlled_factorial')
DATA = MOUNT  # Keep data/checkpoint on Kaggle's read-only input mount; never copy or unpack it into WORK.
assert not (MOUNT / 'controlled_training_input.zip').is_file(), 'Attach the Dataset as an unpacked directory, not controlled_training_input.zip.'
required = [
    DATA / 'checkpoints/model_a/config.json',
    DATA / 'data/processed/vilexnorm_train.jsonl',
    DATA / 'data/processed/vilexnorm_dev.jsonl',
    DATA / 'data/processed/visolex_weak_labeled.jsonl',
    DATA / 'data/processed/visolex_weak_labeled_expanded.jsonl',
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, f'Missing input files: {missing}'
assert not (DATA / 'data/processed/vilexnorm_test.jsonl').exists(), 'Do not attach Test.'
assert not (DATA / 'outputs/evaluation').exists(), 'Do not attach historical evaluation outputs.'
def show_disk(label):
    usage = shutil.disk_usage('/kaggle/working')
    print(f'{label}: free={usage.free / 2**30:.2f} GiB, used={usage.used / 2**30:.2f} GiB')

WORK.mkdir(parents=True, exist_ok=True)
show_disk('Before training')
print('Dev-only input inventory verified; input remains on /kaggle/input.')

In [ ]:
!python -m pytest tests/training/test_controlled_experiments.py -q
# Source provenance is read from the clean Git checkout (REPO); input checksums are read from the private Kaggle Dataset (DATA).
!python -m scripts.controlled_experiments freeze-protocol --source-root {REPO} --data-root {DATA} --config {REPO}/configs/controlled_factorial_config.json --cmax-config {REPO}/configs/c_max20_early_stopping_config.json --output {WORK}/protocol.json

In [ ]:
# Build both paired manifests for each frozen seed.
for seed in (2026, 2126, 2226):
    subprocess.run([
        'python', '-m', 'scripts.controlled_experiments', 'build-factorial',
        '--data-root', str(DATA), '--config', str(REPO/'configs/controlled_factorial_config.json'),
        '--protocol', str(WORK/'protocol.json'), '--seed', str(seed),
        '--output-dir', str(WORK/'runs'),
    ], check=True)

## Smoke gate
Run one 200+200 smoke job. Full runs must use separate work directories and should not use `--smoke-test`.

In [ ]:
SMOKE = WORK / 'smoke_seed_2026_small'
!python -m scripts.controlled_experiments train-factorial --model-a-checkpoint {DATA}/checkpoints/model_a --data-dir {DATA}/data/processed --manifest {WORK}/runs/seed_2026/small/mixture_manifest.json --config {REPO}/configs/controlled_factorial_config.json --work-dir {SMOKE} --smoke-test --no-resume-state
import json
smoke = json.loads((SMOKE/'smoke_test.json').read_text())
assert smoke['passed'] and smoke['composition'] == {'gold': 200, 'pseudo': 200}, smoke
assert smoke['test_inputs_loaded'] is False, smoke
shutil.rmtree(SMOKE)  # Smoke checkpoint is not a scientific artifact and is several GB.
show_disk('After smoke cleanup')
smoke

## Full factorial trajectories — one at a time
Set one `(TARGET_SEED, TARGET_ARM)` below, run the cell, verify cleanup, then change only those two variables for the next cell. The disk-safe mode intentionally does not save AdamW resume state. If Kaggle interrupts a run before cleanup, set `RESET_INTERRUPTED_RUN=True` once to remove that partial trajectory and rerun it from epoch 1. Never reset a run with `cleanup_report.json`.

In [ ]:
TARGET_SEED = 2026
TARGET_ARM = 'small'  # Then run: 2026/expanded, 2126/small, 2126/expanded, 2226/small, 2226/expanded.
RESET_INTERRUPTED_RUN = False
assert TARGET_SEED in (2026, 2126, 2226) and TARGET_ARM in ('small', 'expanded')
run_dir = WORK / 'runs' / f'seed_{TARGET_SEED}' / TARGET_ARM
manifest_path = run_dir / 'mixture_manifest.json'
assert manifest_path.is_file(), f'Missing frozen manifest: {manifest_path}'
if (run_dir/'cleanup_report.json').is_file():
    raise RuntimeError(f'{run_dir} is already completed and cleaned; choose the next trajectory.')
if RESET_INTERRUPTED_RUN:
    assert not (run_dir/'cleanup_report.json').exists(), 'Never reset a completed run.'
    for name in ('state', 'best', 'horizon_3', 'horizon_8', 'train_config.json'):
        path = run_dir/name
        if path.is_dir(): shutil.rmtree(path)
        elif path.exists(): path.unlink()
show_disk(f'Before seed={TARGET_SEED}, arm={TARGET_ARM}')
subprocess.run([
    'python', '-m', 'scripts.controlled_experiments', 'train-factorial',
    '--model-a-checkpoint', str(DATA/'checkpoints/model_a'),
    '--data-dir', str(DATA/'data/processed'),
    '--manifest', str(manifest_path),
    '--config', str(REPO/'configs/controlled_factorial_config.json'),
    '--work-dir', str(run_dir),
    '--no-resume-state',
], check=True)
required = [
    run_dir/'train_config.json',
    *[run_dir/f'horizon_{horizon}'/name for horizon in (3, 8) for name in ('selection.json', 'dev_predictions.jsonl', 'dev_metrics.json', 'terminal_dev_predictions.jsonl', 'terminal_dev_metrics.json')],
]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, f'Run completed without required scientific artifacts: {missing}'
subprocess.run(['python', '-m', 'scripts.controlled_experiments', 'cleanup-factorial-run', '--work-dir', str(run_dir)], check=True)
assert not (run_dir/'state').exists() and not (run_dir/'best').exists()
show_disk(f'After cleanup seed={TARGET_SEED}, arm={TARGET_ARM}')

In [ ]:
!python -m scripts.controlled_experiments summarize-factorial --input-root {WORK}/runs --output {WORK}/factorial_summary.json
summary = json.loads((WORK/'factorial_summary.json').read_text())
assert summary['seeds'] == 3 and summary['test_metrics_used'] is False
summary['effects']

In [ ]:
# Every completed trajectory was already cleaned after verification. The archive contains only small reproducibility artifacts.
assert len(list((WORK/'runs').glob('seed_*/*/cleanup_report.json'))) == 6, 'Run all six trajectories before archiving.'
shutil.make_archive('/kaggle/working/controlled_factorial_artifacts', 'zip', WORK, 'runs')
shutil.copy2(WORK/'protocol.json', '/kaggle/working/controlled_factorial_protocol.json')
shutil.copy2(WORK/'factorial_summary.json', '/kaggle/working/controlled_factorial_summary.json')
print('Download controlled_factorial_artifacts.zip, controlled_factorial_protocol.json, and controlled_factorial_summary.json.')